# Classification by technical field (`TLS230_APPLN_TECHN_FIELD`)
The `TLS230_APPLN_TECHN_FIELD` table provides the degree to which each application belongs to one or more technical fields. This table is derived from the `TLS901_TECHN_FIELD_IPC` table and the IPC codes of the applications; therefore, applications without IPC codes are not associated with any technical fields.

Let’s break down the key fields in this table. The first step is to initialise the PATSTAT client and import the table.

In [1]:
from epo.tipdata.patstat import PatstatClient

# Initialize the PATSTAT client
patstat = PatstatClient(env='PROD')

# Access ORM
db = patstat.orm()

# Importing the models
from epo.tipdata.patstat.database.models import TLS230_APPLN_TECHN_FIELD

# Importing sql alchemy package
from sqlalchemy import func

## Key fields in the `TLS230_APPLN_TECHN_FIELD` table
* `APPLN_ID` (primary key): it is the unique identifier for each patent application in the PATSTAT database.
* `TECHN_FIELD_NR`: this attribute uniquely identifies a technology field through a number.
* `WEIGHT`: similar to what we have seen in table `TLS229_APPLN_NACE2`, the weight measures the extent to which each application belongs to one or more technical field. For each application ID, the weight can assume values between 0 and 1. The sum of all weights for a given application always equals 1. The closer the value is to 1, the more the application belongs to the technical field identified.

To better visualize the technical fields associated with each application ID, we can join the `TLS230_APPLN_TECHN_FIELD` table with `TLS901_TECHN_FIELD_IPC` in order to retrieve the names of the technical fields (`TECHN_FIELD`), rather than just their numerical codes. The merge will be performed using `TECHN_FIELD_NR` as the key attribute.

In [ ]:
#Import the two tables TLS901_TECHN_FIELD_IPC and TLS230_APPLN_TECHN_FIELD 
from epo.tipdata.patstat.database.models import TLS230_APPLN_TECHN_FIELD, TLS901_TECHN_FIELD_IPC 

#Join the two table using the TECHN_FIELD_NR to visualize the name of the technical field associated with each application
tech_f = (
    db.query(
        TLS230_APPLN_TECHN_FIELD.appln_id,
        TLS230_APPLN_TECHN_FIELD.techn_field_nr,
        TLS901_TECHN_FIELD_IPC.techn_field,
        TLS230_APPLN_TECHN_FIELD.weight,
        TLS901_TECHN_FIELD_IPC.techn_sector,
        TLS901_TECHN_FIELD_IPC.ipc_maingroup_symbol,
    )

    .join(
        TLS901_TECHN_FIELD_IPC,
        TLS901_TECHN_FIELD_IPC.techn_field_nr== TLS230_APPLN_TECHN_FIELD.techn_field_nr
    )
        .order_by(
          TLS230_APPLN_TECHN_FIELD.appln_id #Order by the application id
        )
        .distinct()  #Add " distinct" to remove duplicate
    .limit(10000)
)

#Show the results as a dataframe
tech_f_df= patstat.df(tech_f)
tech_f_df

Let's retrieve two different applications (21 & 3321) to see to what extent they belong to specific technical fields.

In [ ]:
app_ids = [21, 3321]
tech_f_id = (
    db.query (
        TLS230_APPLN_TECHN_FIELD.appln_id,
        TLS230_APPLN_TECHN_FIELD.techn_field_nr,
        TLS901_TECHN_FIELD_IPC.techn_field,
        TLS230_APPLN_TECHN_FIELD.weight,
    )
    .join(
        TLS901_TECHN_FIELD_IPC,
        TLS901_TECHN_FIELD_IPC.techn_field_nr== TLS230_APPLN_TECHN_FIELD.techn_field_nr
    )
    .filter(
        TLS230_APPLN_TECHN_FIELD.appln_id.in_(app_ids)
    )
    .order_by(
         TLS230_APPLN_TECHN_FIELD.appln_id
    )
    
    
    .distinct()  
    .limit(10000)

    
)

# Show the result as a dataframe
tech_f_id_df = patstat.df(tech_f_id)
tech_f_id_df     
    
        

As we can see from the output, application 21 belongs to both the Electrical machinery, apparatus, energy (technicald field number 1) and the Basic materials chemistry (technicald field number 19) fields.

Notice that in application 21 the weight associated with Basic materials chemistry is greater than the weight associated with Telecommunications (0.666667 > 0.333333), indicating that the application belongs to the Basic materials chemistry technical field to a greater extent.